In [ ]:
# Install or upgrade the Google Generative AI SDK
!pip install --upgrade google-generativeai
!pip install -q google-genai pydantic

In [ ]:
import os
from google import genai
from google.colab import userdata
from pydantic import BaseModel, Field

# 1. Define the JSON schema with strictly updated behavioral descriptions
class ConstraintExtraction(BaseModel):
    strict_constraints: list[str] = Field(
        description="Explicit, non-negotiable rules directly stated in the user prompt. Do not infer anything."
    )
    unspecified_aspects: list[str] = Field(
        description="Deep BEHAVIORAL ambiguities left unaddressed (e.g., edge cases, null/empty inputs, tie-breaking rules, sorting stability, memory mutability, time/space complexity, and error handling). DO NOT include meta-programming details like language or variable names."
    )

# 2. Retrieve your API key from Colab Secrets and initialize the client
api_key = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=api_key)

def extract_constraints(user_prompt: str) -> ConstraintExtraction:
    # 3. Use the robust behavioral-focused system prompt
    system_prompt = (
        "You are a strict requirements analyst for formal software verification.\n"
        "Your task is to analyze an informal user coding prompt and identify:\n"
        "1. 'strict_constraints': Explicit, non-negotiable rules directly stated.\n"
        "2. 'unspecified_aspects': Deep BEHAVIORAL ambiguities left unaddressed.\n\n"
        "CRITICAL RULES FOR UNSPECIFIED ASPECTS:\n"
        "- DO NOT include meta-programming details like programming language, function name, or parameter names.\n"
        "- DO focus strictly on behavioral semantics: edge cases (null/empty inputs), tie-breaking rules, "
        "sorting stability, memory mutability (in-place vs copy), time/space complexity trade-offs, "
        "and error handling strategies."
    )

    # 4. Call the modern Gemini API passing the Pydantic schema
    # Note: Using gemini-flash-latest as it is generally stable for structured JSON tasks
    response = client.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=[system_prompt, f"User Prompt: '{user_prompt}'"],
        config={
            "response_mime_type": "application/json",
            "response_schema": ConstraintExtraction,
            "temperature": 0.0 # Strict zero temperature for deterministic parsing
        }
    )

    # 5. Parse the returned JSON text into your Pydantic object
    return ConstraintExtraction.model_validate_json(response.text)

# --- Example Usage ---
informal_input = "Write a function that merges two arrays of integers, removes duplicates, and returns it."
extracted = extract_constraints(informal_input)

print("📌 STRICT CONSTRAINTS:")
for constraint in extracted.strict_constraints:
    print(f"  • {constraint}")

print("\n❓ UNSPECIFIED ASPECTS (Behavioral):")
for aspect in extracted.unspecified_aspects:
    print(f"  • {aspect}")

📌 STRICT CONSTRAINTS:
  • Merge two arrays of integers
  • Remove duplicates
  • Return the result

❓ UNSPECIFIED ASPECTS (Behavioral):
  • Handling of null, undefined, or empty array inputs
  • Preservation of original input array mutability versus returning a new array
  • Ordering or sorting requirements of the resulting array (e.g., insertion order, ascending, or descending)
  • Handling of integer overflow or out-of-range values
  • Tie-breaking behavior when duplicate elements appear across both arrays


In [ ]:
import time

# List of 20 test prompts (simple, complex, ambiguous, and edge-case heavy)
test_prompts = [
    # 1. Basic Algorithmic
    "Write a function to check if a string is a palindrome.",

    # 2. Data Structure & Caching
    "Design a Least Recently Used (LRU) cache with a fixed capacity N.",

    # 3. High Ambiguity Business Logic (Tricky)
    "Build a function that flags suspicious bank transactions.",

    # 4. Fuzzy Text / Formatting
    "Create a function that cleans up messy user input text and fixes simple formatting.",

    # 5. Multi-Step Data Pipeline
    "Take a nested JSON object of employees, filter out those earning below $50k who joined before 2020, and return a flattened list.",

    # 6. Real-World Fintech Engineering
    "Given a list of stock trades with timestamps, symbols, prices, and quantities, calculate the volume-weighted average price (VWAP) for each symbol over 5-minute tumbling windows.",

    # 7. Concurrency / I/O Heavy
    "Download 10 images concurrently from a list of URLs and save them to a local directory.",

    # 8. System Design / Rate Limiting
    "Create an in-memory rate limiter that allows a maximum of 100 requests per minute per IP address.",

    # 9. String Extraction / Parsing
    "Extract all valid email addresses and phone numbers from a raw, unstructured text string.",

    # 10. Temporal / Date Math
    "Parse a relative time string like '2 hours 15 mins ago' or '3 days ago' into a standard UTC Unix timestamp.",

    # 11. Domain Calculation
    "Calculate the total daily compound interest accrued on a savings account with tiered interest rates.",

    # 12. Matrix Transformation (In-place constraint)
    "Rotate an N x N 2D matrix representing an image by 90 degrees clockwise in-place.",

    # 13. Tree Navigation
    "Given a binary search tree, find the lowest common ancestor (LCA) of two given nodes.",

    # 14. Security & Cryptography
    "Hash a user password using a salt and verify whether a given raw password matches the stored hash.",

    # 15. Networking / IP Parsing
    "Validate an IPv4 address string and determine if it falls within a private IP address range.",

    # 16. Logical Puzzle / Validation
    "Check if a given 9x9 Sudoku board configuration is valid according to standard rules.",

    # 17. Bitwise Optimization
    "Count the number of set bits in a 64-bit unsigned integer without using built-in language functions.",

    # 18. Streaming / Top-K
    "Find the top 3 highest spending customers from a real-time stream of incoming transaction logs.",

    # 19. Contradictory / Paradoxical Prompt
    "Write a fast search function that finds elements in an unsorted array without scanning through the elements.",

    # 20. Graph Traversals
    "Find the shortest degrees of separation path between two users in a social network graph."
]

print(f"🚀 Running constraint extraction across {len(test_prompts)} prompts...\n")

for idx, prompt in enumerate(test_prompts, 1):
    print(f"==================================================================")
    print(f"TEST #{idx}: \"{prompt}\"")
    print(f"==================================================================")

    try:
        # Calls the function defined in the previous cell
        result = extract_constraints(prompt)

        print("📌 STRICT CONSTRAINTS:")
        for constraint in result.strict_constraints:
            print(f"  • {constraint}")

        print("\n❓ UNSPECIFIED ASPECTS:")
        for aspect in result.unspecified_aspects:
            print(f"  • {aspect}")

    except Exception as e:
        print(f"❌ Error processing prompt: {e}")

    print("\n")
    time.sleep(4)  # Slight delay to respect API rate limits

🚀 Running constraint extraction across 20 prompts...

TEST #1: "Write a function to check if a string is a palindrome."
📌 STRICT CONSTRAINTS:
  • Check if an input string is a palindrome.

❓ UNSPECIFIED ASPECTS:
  • Case sensitivity rules (whether uppercase and lowercase characters are treated as equal)
  • Handling of non-alphanumeric characters and whitespace (whether to ignore or include them in comparison)
  • Behavior for empty strings or single-character strings
  • Handling of null, undefined, or non-string inputs
  • Unicode character normalization and handling of multi-byte characters or accents


TEST #2: "Design a Least Recently Used (LRU) cache with a fixed capacity N."
❌ Error processing prompt: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


TEST #3: "Build a function that flags suspicious bank transactions."
📌 STRICT CONSTRA

In [ ]:
complex_prompts = [
    # 1. The "Hallucination Trap"
    "Build a checkout cart. If they buy 3 apples, apply the discount. Output the final price.",

    # 2. The "Over-Constrained / Impossible" Algorithmic Prompt
    "Write a sorting algorithm that sorts an array of 1 million integers in O(1) time and O(1) space. It must be written in one line of code.",

    # 3. The "Vague Security" Prompt
    "Make my database connection string secure so hackers can't see it when the code runs.",

    # 4. The "Massive Mess"
    "so i need a function that takes a list of users right? but some users dont have emails so just skip them or maybe put a default email idk. then sort them by age but if they are the same age put the taller one first. return it as a dict.",

    # 5. The "Silent State" Problem
    "Write a function generate_id() that returns a new, unique integer every time it is called."
]

print("💥 Running Stress Test on Complex Edge Cases...\n")

for idx, prompt in enumerate(complex_prompts, 1):
    print(f"==================================================================")
    print(f"TEST #{idx}: \"{prompt}\"")
    print(f"==================================================================")

    try:
        result = extract_constraints(prompt)

        print("📌 STRICT CONSTRAINTS:")
        for constraint in result.strict_constraints:
            print(f"  • {constraint}")

        print("\n❓ UNSPECIFIED ASPECTS:")
        for aspect in result.unspecified_aspects:
            print(f"  • {aspect}")

    except Exception as e:
        print(f"❌ Error processing prompt: {e}")

    print("\n")
    time.sleep(50)

💥 Running Stress Test on Complex Edge Cases...

TEST #1: "Build a checkout cart. If they buy 3 apples, apply the discount. Output the final price."
📌 STRICT CONSTRAINTS:
  • Build a checkout cart
  • Apply the discount if exactly or at least 3 apples are bought
  • Output the final price

❓ UNSPECIFIED ASPECTS:
  • How to handle fractional or negative quantities of apples
  • The exact percentage or monetary value of the discount to be applied
  • Whether the discount applies to every set of 3 apples (e.g., buying 6 apples gives two discounts) or just once
  • How other items in the cart besides apples affect the total price and whether they are subject to any rules
  • Currency formatting, rounding rules, and precision for the final price output
  • Error handling behavior for null, missing, or malformed cart inputs
  • Whether the checkout cart operates on an immutable data structure or modifies the input state in place


TEST #2: "Write a sorting algorithm that sorts an array of 1 m

In [ ]:
# 20 Standard Algorithmic Prompts with High Ambiguity / Multiple Intents
algorithmic_prompts = [
    # 1. Two Sum / Array Indexing
    "Write a function to find two numbers in an array that sum up to a given target.",

    # 2. Binary Search / Duplicates
    "Search for a target value in a sorted array and return its location.",

    # 3. Graph Traversal (BFS vs DFS / Path vs Reachability)
    "Given a graph, check if there is a path from node A to node B.",

    # 4. K-Way Merge / Deduplication
    "Merge multiple sorted integer arrays into a single sorted array.",

    # 5. Sliding Window / Longest Substring
    "Find the longest substring without repeating characters in a string.",

    # 6. Weighted Shortest Path / Multi-Objective Dijkstra
    "Find the cheapest route between two cities given road distances and toll costs.",

    # 7. Tree Traversal Ordering
    "Return the nodes of a binary tree level by level.",

    # 8. Knapsack Problem (Fractional vs 0/1 / Items vs Max Value)
    "Given a set of items with weights and values, maximize total value under a weight limit W.",

    # 9. Topological Sort / Dependency Resolution
    "Order a list of tasks where some tasks depend on the completion of others.",

    # 10. Quickselect / K-th Element
    "Find the K-th largest number in an unsorted list.",

    # 11. Matrix Spiral Traversal
    "Traverse a 2D grid in a spiral order and return the visited elements.",

    # 12. String Compression (Run-Length Encoding)
    "Compress a string by replacing repeated consecutive characters with their count.",

    # 13. Stack / Valid Parentheses
    "Check if a string containing brackets is properly balanced.",

    # 14. Maximum Subarray Sum (Kadane's Algorithm)
    "Find the contiguous subarray that has the largest sum.",

    # 15. Prefix Search / Trie Autocomplete
    "Build a word suggestion function that returns matching words given a search prefix.",

    # 16. Linked List Cycle Detection
    "Determine if a linked list contains a cycle.",

    # 17. Coin Change / Dynamic Programming
    "Find the minimum number of coins needed to make a specific target amount.",

    # 18. Stream Median Finder
    "Design a system that continuously receives numbers and returns the current median.",

    # 19. Interval Merging
    "Merge all overlapping time intervals in a list.",

    # 20. Lowest Common Ancestor (LCA)
    "Find the lowest common ancestor of two nodes in a tree."
]

# Run the extraction on these prompts
print("🚀 Running behavioral constraint extraction on 20 algorithmic prompts...\n")

for idx, prompt in enumerate(algorithmic_prompts, 1):
    print(f"==================================================================")
    print(f"TEST #{idx}: \"{prompt}\"")
    print(f"==================================================================")

    try:
        result = extract_constraints(prompt)

        print("📌 STRICT CONSTRAINTS:")
        for constraint in result.strict_constraints:
            print(f"  • {constraint}")

        print("\n❓ UNSPECIFIED ASPECTS (Behavioral):")
        for aspect in result.unspecified_aspects:
            print(f"  • {aspect}")

    except Exception as e:
        print(f"❌ Error processing prompt: {e}")

    print("\n")
    time.sleep(12)  # 12-second delay to strictly enforce max 5 RPM on free tier

🚀 Running behavioral constraint extraction on 20 algorithmic prompts...

TEST #1: "Write a function to find two numbers in an array that sum up to a given target."
📌 STRICT CONSTRAINTS:
  • Find two numbers in an array that sum up to a given target

❓ UNSPECIFIED ASPECTS (Behavioral):
  • How to handle inputs with fewer than two elements
  • What to return if no such pair exists
  • Whether multiple valid pairs exist and whether to return all of them, the first found, or any arbitrary pair
  • Whether the same element index can be used twice to reach the target sum
  • Whether the input array can be modified in place or must remain immutable
  • How to handle invalid input types, null, or out-of-range integer values
  • Expected time and space complexity constraints


TEST #2: "Search for a target value in a sorted array and return its location."
📌 STRICT CONSTRAINTS:
  • Search for a target value in a sorted array
  • Return the location of the target value

❓ UNSPECIFIED ASPECTS (Beh

In [ ]:
mbpp_prompts = [
    "Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix cost[][] and a position (m, n) in cost[][].",
    "Write a function to find the similar elements from the given two tuple lists.",
    "Write a python function to identify non-prime numbers.",
    "Write a function to find the largest integers from a given list of numbers using heap queue algorithm.",
    "Write a function to find the number of ways to fill it with 2 x 1 dominoes for the given 3 x n board.",
    "Write a python function to check whether the two numbers differ at one bit position only or not.",
    "Write a function to find all words which are at least 4 characters long in a string by using regex.",
    "Write a function to find squares of individual elements in a list using lambda function.",
    "Write a python function to find the minimum number of rotations required to get the same string.",
    "Write a function to get the n smallest items from a dataset.",
    "Write a python function to remove first and last occurrence of a given character from the string.",
    "Write a function to sort a given matrix in ascending order according to the sum of its rows.",
    "Write a function to count the most common words in a dictionary.",
    "Write a python function to find the volume of a triangular prism.",
    "Write a function to split a string at lowercase letters."
]

print(f"🚀 Running behavioral constraint extraction on {len(mbpp_prompts)} examples...\n")

for idx, prompt in enumerate(mbpp_prompts, 1):
    print(f"==================================================================")
    print(f"TEST #{idx}: \"{prompt}\"")
    print(f"==================================================================")

    max_retries = 3
    for attempt in range(max_retries):
        try:
            result = extract_constraints(prompt)

            print("📌 STRICT CONSTRAINTS:")
            for constraint in result.strict_constraints:
                print(f"  • {constraint}")

            print("\n❓ UNSPECIFIED ASPECTS (Behavioral):")
            for aspect in result.unspecified_aspects:
                print(f"  • {aspect}")

            break

        except Exception as e:
            error_msg = str(e)
            if "503" in error_msg or "UNAVAILABLE" in error_msg:
                wait_time = 5 * (attempt + 1)
                print(f"⚠️ Server busy (503). Retrying attempt {attempt + 1}/{max_retries} in {wait_time} seconds...")
                time.sleep(wait_time)
            elif "429" in error_msg:
                 print(f"❌ Rate limit hit (429). Please check your quotas.")
                 time.sleep(12)
                 break
            else:
                print(f"❌ Unhandled Error: {error_msg}")
                break

    print("\n")
    time.sleep(12)

🚀 Running behavioral constraint extraction on 15 examples...

TEST #1: "Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix cost[][] and a position (m, n) in cost[][]."
📌 STRICT CONSTRAINTS:
  • Find the minimum cost path to reach (m, n) from (0, 0)
  • Use the given cost matrix cost[][]
  • Target position is (m, n)

❓ UNSPECIFIED ASPECTS (Behavioral):
  • Allowed directions of movement (e.g., right, down, diagonally)
  • Behavior when input coordinates (m, n) are out of bounds of the cost matrix
  • Behavior when the cost matrix is empty, null, or has non-positive dimensions
  • How to handle negative cost values and potential integer overflow
  • Tie-breaking behavior when multiple paths share the exact same minimum cost
  • Whether the cost matrix should be mutated in-place during computation
  • Return value representation for unreachable destinations or invalid inputs


TEST #2: "Write a function to find the similar elements from t

In [ ]:
import google.generativeai as genai

print('Available Gemini models:')
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

Available Gemini models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview

In [ ]:
# After installing, restart the runtime (Runtime > Restart runtime) to apply the changes.
# Then, re-run all cells.

In [ ]:
import time
import os
from google import genai
from google.colab import userdata
from pydantic import BaseModel, Field

# =====================================================================
# 1. PYDANTIC SCHEMAS (Defining the Structured Outputs)
# =====================================================================

class ConstraintExtraction(BaseModel):
    strict_constraints: list[str] = Field(
        description="Explicit, non-negotiable rules directly stated in the user prompt."
    )
    unspecified_aspects: list[str] = Field(
        description="Deep BEHAVIORAL ambiguities left unaddressed (e.g., edge cases, tie-breaking, nulls)."
    )

class FormalSpecification(BaseModel):
    intent_title: str = Field(description="A short, descriptive title for this specific interpretation.")
    assumptions_made: list[str] = Field(description="Explicit statements detailing how the unspecified aspects were resolved.")
    formal_english_rules: list[str] = Field(description="A bulleted list of highly formal English rules defining the exact expected behavior.")

class AmbiguityExpansion(BaseModel):
    specifications: list[FormalSpecification] = Field(
        description="A list of exactly 3 distinct, non-contradictory formal specifications.",
        min_items=3, max_items=3
    )

# =====================================================================
# 2. INITIALIZE CLIENT
# =====================================================================
api_key = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=api_key)

# =====================================================================
# 3. PIPELINE FUNCTIONS
# =====================================================================

def extract_constraints(user_prompt: str) -> ConstraintExtraction:
    system_prompt = (
        "You are a strict requirements analyst for formal software verification.\n"
        "Analyze the informal user prompt and identify strict constraints and behavioral ambiguities.\n"
        "DO NOT include meta-programming details like language or variable names."
    )
    response = client.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=[system_prompt, f"User Prompt: '{user_prompt}'"],
        config={"response_mime_type": "application/json", "response_schema": ConstraintExtraction, "temperature": 0.0}
    )
    return ConstraintExtraction.model_validate_json(response.text)

def generate_formal_specs(original_prompt: str, step1_data: ConstraintExtraction) -> AmbiguityExpansion:
    system_prompt = (
        "You are an expert technical product manager and systems architect.\n"
        "Take the informal user prompt and the constraint analysis, and generate exactly 3 DISTINCT formal specifications.\n"
        "Do NOT contradict the strict constraints. Resolve the unspecified aspects uniquely for each specification."
    )
    combined_input = (
        f"--- ORIGINAL PROMPT ---\n{original_prompt}\n\n"
        f"--- STEP 1 ANALYSIS ---\n{step1_data.model_dump_json(indent=2)}"
    )
    response = client.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=[system_prompt, combined_input],
        config={"response_mime_type": "application/json", "response_schema": AmbiguityExpansion, "temperature": 0.4}
    )
    return AmbiguityExpansion.model_validate_json(response.text)

# =====================================================================
# 4. DATASET (Extracted 'text' only)
# =====================================================================
mbpp_prompts = [

    "Write a function to find m number of multiples of n.",
    "Write a function to find the first duplicate element in a given array of integers.",
    "Write a python function to find the maximum sum of elements of list in a list of lists.",
    "Write a function to convert the given binary number to its decimal equivalent.",
    "Write a python function to find the product of non-repeated elements in a given array.",
    "Write a function to check if the given tuple list has all k elements.",
    "Write a python function to remove all digits from a list of strings.",

]

# =====================================================================
# 5. EXECUTION & FILE WRITING LOOP
# =====================================================================
output_filename = "btp_pipeline_results.md"

print(f"🚀 Running End-to-End Pipeline on {len(mbpp_prompts)} prompts...")
print(f"📁 Output will be saved to: {output_filename}\n")

# Open file in write mode
with open(output_filename, "w", encoding="utf-8") as file:
    file.write("# BTP Intent Formalization Pipeline - Evaluation Results\n\n")

    for idx, prompt in enumerate(mbpp_prompts, 1):
        print(f"Processing Test #{idx}/{len(mbpp_prompts)}...")

        max_retries = 3
        for attempt in range(max_retries):
            try:
                # Run Step 1
                step1_res = extract_constraints(prompt)
                time.sleep(2) # Brief pause between sequential API calls

                # Run Step 2
                step2_res = generate_formal_specs(prompt, step1_res)

                # --- FORMAT AND WRITE TO FILE ---
                file.write(f"## Test #{idx}\n")
                file.write(f"**Original Prompt:** `{prompt}`\n\n")

                file.write("### Step 1: Constraint & Ambiguity Extraction\n")
                file.write("**Strict Constraints:**\n")
                for c in step1_res.strict_constraints:
                    file.write(f"- {c}\n")

                file.write("\n**Unspecified Aspects (Behavioral):**\n")
                for u in step1_res.unspecified_aspects:
                    file.write(f"- {u}\n")

                file.write("\n### Step 2: Formal Specification Generation (3 Distinct Intents)\n")
                for i, spec in enumerate(step2_res.specifications, 1):
                    file.write(f"#### Option {i}: {spec.intent_title}\n")
                    file.write("**Assumptions Made:**\n")
                    for a in spec.assumptions_made:
                        file.write(f"- {a}\n")
                    file.write("**Formal Rules:**\n")
                    for r in spec.formal_english_rules:
                        file.write(f"- {r}\n")
                    file.write("\n")

                file.write("---\n\n")

                print(f"✅ Successfully wrote Test #{idx} to file.")
                break # Break out of retry loop on success

            except Exception as e:
                error_msg = str(e)
                if "503" in error_msg or "UNAVAILABLE" in error_msg:
                    wait_time = 5 * (attempt + 1)
                    print(f"⚠️ Server busy (503). Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                elif "429" in error_msg:
                     print(f"❌ Rate limit hit (429). Sleeping for 15s...")
                     time.sleep(15)
                else:
                    print(f"❌ Unhandled Error on Test #{idx}: {error_msg}")
                    file.write(f"## Test #{idx}\n**Error Processing:** {error_msg}\n\n---\n\n")
                    break

        # Enforce rate limit pause before the next prompt
        time.sleep(20)

print(f"\n🎉 Pipeline complete! You can now download '{output_filename}' from the Colab file explorer.")

/tmp/ipykernel_619/3302592351.py:25: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  specifications: list[FormalSpecification] = Field(
/tmp/ipykernel_619/3302592351.py:25: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  specifications: list[FormalSpecification] = Field(


🚀 Running End-to-End Pipeline on 7 prompts...
📁 Output will be saved to: btp_pipeline_results.md

Processing Test #1/7...
✅ Successfully wrote Test #1 to file.
Processing Test #2/7...
✅ Successfully wrote Test #2 to file.
Processing Test #3/7...
✅ Successfully wrote Test #3 to file.
Processing Test #4/7...
✅ Successfully wrote Test #4 to file.
Processing Test #5/7...
✅ Successfully wrote Test #5 to file.
Processing Test #6/7...
✅ Successfully wrote Test #6 to file.
Processing Test #7/7...
✅ Successfully wrote Test #7 to file.

🎉 Pipeline complete! You can now download 'btp_pipeline_results.md' from the Colab file explorer.
